# Deep Research Agent - Startup Funding Readiness Evaluator

## Project Overview
This Deep Research Agent automates the evaluation of a startup's pitch deck or business plan against current venture capital funding criteria. It combines real-time web research on VC expectations, market trends, and investment benchmarks with AI-powered analysis to provide a comprehensive readiness assessment — helping founders understand their strengths, gaps, and what to improve before approaching investors.

In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
from pypdf import PdfReader
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown
from openai import OpenAI

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

## Data Models

In [ ]:
class WebSearchItem(BaseModel):
    reason: str = Field(
        description="Your reasoning for why this search is important to the query."
    )
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description="A list of web searches to perform to best answer the query."
    )


class PitchDeck(BaseModel):
    content: str = Field(
        description="The extracted text from the pitch deck / business plan PDF"
    )
    source_path: str = Field(
        description="The file path or URL where the PDF was loaded from"
    )


class ReportData(BaseModel):
    short_summary: str = Field(
        description="A short 2-3 sentence summary of the findings."
    )
    markdown_report: str = Field(description="The final report in markdown format")
    follow_up_questions: list[str] = Field(
        description="Suggested areas to research or improve further"
    )

## Read the Pitch Deck / Business Plan

In [ ]:
def read_pitch_deck(path: str) -> str:
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

## Search Agent
Performs individual web searches and summarises results concisely.

In [ ]:
SEARCH_INSTRUCTIONS = (
    "You are a research assistant. Given a search term, you search the web for that term and "
    "produce a concise summary of the results. The summary must be 2-3 paragraphs and less than 300 "
    "words. Capture the main points. Write succinctly, no need to have complete sentences or good "
    "grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the "
    "essence and ignore any fluff. Do not include any additional commentary other than the summary itself."
)

search_agent = Agent(
    name="Search agent",
    instructions=SEARCH_INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

## Planner Agent
Plans which searches to run to gather current VC funding criteria.

In [ ]:
HOW_MANY_SEARCHES = 5

PLANNER_INSTRUCTIONS = (
    f"You are a helpful research assistant specializing in startup fundraising and venture capital. "
    f"Given a query about startup funding readiness, come up with a set of web searches to find the "
    f"most current and accurate information. Focus on: current VC investment criteria, what top investors "
    f"look for in pitch decks, common reasons startups get rejected, market-sizing best practices, "
    f"and competitive analysis frameworks. "
    f"Output {HOW_MANY_SEARCHES} search terms that will help gather up-to-date, actionable intelligence."
)

planner_agent = Agent(
    name="PlannerAgent",
    instructions=PLANNER_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

## Email Tool & Agent

In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send an email with the given subject and HTML body."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email(os.environ.get("FROM_EMAIL", "you@example.com"))
    to_email = To(os.environ.get("TO_EMAIL", "you@example.com"))
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"


EMAIL_INSTRUCTIONS = (
    "You are able to send a nicely formatted HTML email based on a detailed report. "
    "You will be provided with a detailed report. You should use your tool to send one email, providing the "
    "report converted into clean, well presented HTML with an appropriate subject line."
)

email_agent = Agent(
    name="Email agent",
    instructions=EMAIL_INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)

## Report Writer Agent
Synthesises research into a structured funding-readiness report.

In [ ]:
WRITER_INSTRUCTIONS = (
    "You are a senior startup advisor tasked with writing a cohesive Funding Readiness Report. "
    "You will be provided with the original query, the startup's pitch deck content, and research "
    "gathered by a research assistant about current VC criteria and market conditions.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow. Then, generate the report and return that as your final output.\n"
    "The report MUST include these sections:\n"
    "1. Executive Summary\n"
    "2. Team Assessment\n"
    "3. Market Opportunity Analysis\n"
    "4. Product / Technology Evaluation\n"
    "5. Business Model & Traction\n"
    "6. Competitive Landscape\n"
    "7. Financial Projections Review\n"
    "8. Pitch Deck Quality Assessment\n"
    "9. Overall Funding Readiness Score (1-10 with justification)\n"
    "10. Actionable Recommendations\n\n"
    "The final output should be in markdown format. Aim for 5-10 pages of content, at least 1500 words."
)

writer_agent = Agent(
    name="WriterAgent",
    instructions=WRITER_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

## Core Pipeline Functions

In [ ]:
async def plan_searches(query: str):
    """Use the planner agent to decide which searches to run."""
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output


async def perform_searches(search_plan: WebSearchPlan):
    """Run all planned searches concurrently."""
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    """Run a single web search via the search agent."""
    input_text = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_text)
    return result.final_output


async def evaluate_startup(search_results, pitch_text: str):
    """Evaluate the startup pitch against current VC funding criteria."""
    system_prompt = (
        "You are an experienced venture capital analyst evaluating a startup's pitch deck "
        "against current industry funding criteria. Your task is to analyze the pitch deck content "
        "and assess how well it aligns with what top-tier VCs are looking for. "
        "Provide a detailed, objective evaluation highlighting strengths, red flags, and "
        "concrete recommendations. Be thorough, data-driven, and constructive."
    )
    system_prompt += f"\n\n## Current VC Funding Criteria & Market Intelligence:\n{search_results}\n\n"
    system_prompt += f"## Startup Pitch Deck Content:\n{pitch_text}\n\n"

    user_message = (
        "Please analyze this startup's pitch deck against current VC funding criteria. "
        "In your evaluation:\n"
        "1. Assess the team's credibility and relevant experience\n"
        "2. Evaluate the market opportunity sizing and validation\n"
        "3. Analyze the product-market fit signals and traction metrics\n"
        "4. Review the business model and unit economics\n"
        "5. Assess competitive positioning and defensibility\n"
        "6. Evaluate financial projections for realism\n"
        "7. Rate the overall pitch deck quality and storytelling\n"
        "8. Provide a funding readiness score from 1-10 with justification\n"
        "9. List the top 5 things to fix before approaching investors"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content


async def write_report(query: str, evaluation: str, pitch_text: str):
    """Use the writer agent to produce the final funding readiness report."""
    print("Writing report...")
    input_text = (
        f"Original query: {query}\n\n"
        f"Pitch deck content:\n{pitch_text}\n\n"
        f"Evaluation and research findings:\n{evaluation}"
    )
    result = await Runner.run(writer_agent, input_text)
    print("Finished writing report")
    return result.final_output


async def send_report_email(report: ReportData):
    """Use the email agent to send the report."""
    print("Sending email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

## Run the Pipeline

Place your startup pitch deck PDF at `pitch_deck/pitch.pdf` (relative to this notebook), or change the path below.

In [ ]:
pitch_deck_path = "pitch_deck/pitch.pdf"

query = (
    "What are the key criteria that top-tier venture capital firms use to evaluate "
    "early-stage startups for seed and Series A funding in 2025-2026? "
    "I want to assess whether a startup's pitch deck meets current investor expectations."
)

with trace("Startup Funding Readiness Evaluation"):
    print("Starting startup funding readiness evaluation...")

    pitch_text = read_pitch_deck(pitch_deck_path)
    print(f"Loaded pitch deck ({len(pitch_text)} characters)")

    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)

    evaluation = await evaluate_startup(search_results, pitch_text)
    print("\n--- Preliminary Evaluation ---")
    print(evaluation)

    report = await write_report(query, evaluation, pitch_text)

    print("\n--- Final Report ---")
    display(Markdown(report.markdown_report))

    print("\n--- Summary ---")
    print(report.short_summary)

    print("\n--- Follow-up Questions ---")
    for q in report.follow_up_questions:
        print(f"  • {q}")

    await send_report_email(report)
    print("\nDone!")